# Baseline 2 — Widar_CNN_GRU (SenseFi / Widar3.0 TPAMI)

**Source:** Yang et al., *SenseFi: A Library and Benchmark on Deep-Learning-Empowered
WiFi Human Sensing*, Patterns (Cell Press), 2023.  
GitHub: https://github.com/xyanchen/WiFi-CSI-Sensing-Benchmark — `widar_model.py`

**Architecture (exact from SenseFi, one adaptation):**
```
For each of T frames independently:
  Input frame  (B*T, 1, 20, 20)
  Conv2d(1→8,  k=6, stride=2) → ReLU         → (B*T,  8, 8, 8)
  Conv2d(8→16, k=3, stride=1) → ReLU          → (B*T, 16, 6, 6)
  MaxPool2d(2)                                  → (B*T, 16, 3, 3)
  Flatten → Linear(144→64) → ReLU → Dropout → Linear(64→64) → ReLU

Temporal modelling:
  Reshape → (T, B, 64) → GRU(64, hidden=128) → ht: (B, 128)
  Dropout → Linear(128 → num_classes)
```

**Adaptation:** SenseFi uses T=22. Our preprocessing uses T=20.  
We set `T_FRAMES=20` in the forward pass. All layers identical.

**Protocol:** 3-fold Leave-One-Room-Out (LORO), standard-6 gestures.  
**Runtime:** ~15–25 min total on Colab **T4 GPU**.

## 1 — Colab Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO_DIR = '/content/pi-ssl'
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Fatima112299/pi-ssl.git {REPO_DIR}
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 2 — Configuration

In [ ]:
NPZ_PATH      = '/content/drive/MyDrive/pi-ssl/preprocessed.npz'

EPOCHS        = 100
BATCH_SIZE    = 64
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 2
RANDOM_SEED   = 42
LABELED_RATIO = 0.25
NUM_CLASSES   = 6
T_FRAMES      = 20   # our T; SenseFi original uses 22

print(f'NPZ_PATH : {NPZ_PATH}')
print(f'T_FRAMES : {T_FRAMES}  (SenseFi original: 22 — adapted to our T_TARGET=20)')

## 3 — Imports

In [ ]:
import sys, os
REPO_DIR = '/content/pi-ssl'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if not os.path.exists(REPO_DIR):
    raise RuntimeError('Repo not found — run the clone cell first.')

import time, json, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report

from src.data.bvp_dataset import load_npz, BVPDataset
from src.data.splits import make_loeo_splits
from src.data.widar3_dataset import ID_TO_GESTURE

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU. Enable T4 GPU in Runtime → Change runtime type.')
print('All imports OK.')

## 4 — Load Data

In [ ]:
print('Loading preprocessed.npz ...')
t0 = time.time()
npz_data, file_list = load_npz(NPZ_PATH)
print(f'  Samples   : {len(file_list)}')
print(f'  BVP shape : {npz_data["bvp"].shape}  {npz_data["bvp"].dtype}')
print(f'  Load time : {time.time()-t0:.1f}s')

## 5 — Widar_CNN_GRU Model

Exact architecture from SenseFi `widar_model.py`.

**Two adaptations from original:**
1. `T_FRAMES=20` instead of 22 — matches our preprocessing
2. Removed `nn.Softmax` from classifier — SenseFi included it for inference;
   we use `nn.CrossEntropyLoss` which expects raw logits, so Softmax is omitted
   during training (standard PyTorch practice)

In [ ]:
class Widar_CNN_GRU(nn.Module):
    """
    CNN-GRU hybrid from SenseFi (Yang et al., Patterns 2023).

    Each BVP frame (20×20) is encoded independently by a small CNN,
    producing a 64-dim feature vector. The sequence of T frame features
    is then processed by a GRU to capture temporal dynamics.

    Original SenseFi forward:
        x = x.view(batch_size*22, 1, 20, 20)  # T=22
        ...
        x = x.view(-1, 22, 64)

    Our adaptation:
        x = x.view(batch_size*20, 1, 20, 20)  # T=20
        ...
        x = x.view(-1, 20, 64)

    Source: https://github.com/xyanchen/WiFi-CSI-Sensing-Benchmark/blob/main/widar_model.py
    """
    def __init__(self, num_classes=6, t_frames=20):
        super().__init__()
        self.t_frames = t_frames

        # Per-frame spatial encoder (identical to SenseFi)
        self.encoder = nn.Sequential(
            nn.Conv2d(1,  8,  kernel_size=6, stride=2),  # (1,20,20) → (8,8,8)
            nn.ReLU(),
            nn.Conv2d(8, 16,  kernel_size=3, stride=1),  # (8,8,8)  → (16,6,6)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (16,6,6) → (16,3,3)
        )
        # 16 * 3 * 3 = 144
        self.fc = nn.Sequential(
            nn.Linear(16 * 3 * 3, 64),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(64, 64),
            nn.ReLU(),
        )

        # Temporal GRU over T frame features
        self.gru = nn.GRU(64, 128, num_layers=1)

        # Classifier — Softmax removed; use CrossEntropyLoss with raw logits
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x from BVPDataset: (B, 1, T, 20, 20)
        batch_size = x.size(0)

        # Squeeze channel, reshape to process all frames at once
        x = x.squeeze(1)                              # (B, T, 20, 20)
        x = x.view(batch_size * self.t_frames, 1, 20, 20)  # (B*T, 1, 20, 20)

        # Per-frame CNN encoding
        x = self.encoder(x)                           # (B*T, 16, 3, 3)
        x = x.view(batch_size * self.t_frames, -1)    # (B*T, 144)
        x = self.fc(x)                                # (B*T, 64)

        # Reshape for GRU: (T, B, 64)
        x = x.view(batch_size, self.t_frames, 64)     # (B, T, 64)
        x = x.permute(1, 0, 2)                        # (T, B, 64)

        # GRU: take last hidden state
        _, ht = self.gru(x)                           # ht: (1, B, 128)

        # Classify from final hidden state
        return self.classifier(ht[-1])                # (B, num_classes)


# Shape check
set_seed(RANDOM_SEED)
_m = Widar_CNN_GRU(num_classes=NUM_CLASSES, t_frames=T_FRAMES).to(DEVICE)
_x = torch.zeros(2, 1, 20, 20, 20).to(DEVICE)   # BVPDataset format: (B,1,T,H,W)
_o = _m(_x)
print(f'Widar_CNN_GRU output shape : {_o.shape}  (expect [2, {NUM_CLASSES}])')
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Trainable params           : {n_params:,}')
del _m, _x, _o

## 6 — Training Utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_correct, total_n = 0.0, 0, 0
    for bvp, labels in loader:
        bvp    = bvp.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(bvp)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * len(labels)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_n       += len(labels)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for bvp, labels in loader:
        preds = model(bvp.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


def run_fold_setting(tag, train_files, test_files, npz_data, device,
                     epochs, batch_size, lr, weight_decay, seed, num_classes, t_frames):
    set_seed(seed)
    print(f'\n  [{tag}]  train={len(train_files)}  test={len(test_files)}')

    train_ds = BVPDataset(train_files, npz_data)
    test_ds  = BVPDataset(test_files,  npz_data)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'),
                              drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size*2, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))

    model     = Widar_CNN_GRU(num_classes=num_classes, t_frames=t_frames).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    t_start = time.time()
    for epoch in range(1, epochs + 1):
        loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        scheduler.step()
        if epoch % 20 == 0 or epoch == 1:
            print(f'    Epoch {epoch:>3}/{epochs}  loss={loss:.4f}  '
                  f'train_acc={100*train_acc:.1f}%  elapsed={time.time()-t_start:.0f}s')

    train_time = time.time() - t_start
    y_pred, y_test = evaluate(model, test_loader, device)
    acc = accuracy_score(y_test, y_pred) * 100
    print(f'    Train time : {train_time:.1f}s')
    print(f'    Test acc   : {acc:.2f}%')

    labels_present = sorted(set(y_test.tolist()))
    class_names    = [ID_TO_GESTURE[i] for i in labels_present]
    report = classification_report(y_test, y_pred, labels=labels_present,
                                   target_names=class_names, digits=3, zero_division=0)
    print('    Per-class breakdown:')
    for line in report.strip().split('\n'):
        print(f'      {line}')

    return {'accuracy': acc, 'train_time': train_time, 'y_pred': y_pred, 'y_test': y_test}


print('Utilities defined.')

## 7 — Run 3-Fold LORO Evaluation

> **Runtime:** ~5–8 min per model on T4 GPU. 6 models ≈ 30–50 min total.

In [ ]:
FOLD_ROOMS   = {0: 'Office', 1: 'Classroom', 2: 'Hall'}
fold_results = {}
t_total      = time.time()

for fold in range(3):
    print(f'\n{"="*62}')
    print(f'FOLD {fold}  —  test room: {FOLD_ROOMS[fold]}')
    print(f'{"="*62}')

    _, labeled, unlabeled, test = make_loeo_splits(
        bvp_root=None, fold=fold,
        labeled_ratio=LABELED_RATIO, seed=RANDOM_SEED,
        file_list=file_list
    )
    full_train = labeled + unlabeled

    print(f'  Labeled train (25%) : {len(labeled)}')
    print(f'  Full train   (100%) : {len(full_train)}')
    print(f'  Test                : {len(test)}')

    fold_results[fold] = {}

    fold_results[fold]['25%_labeled'] = run_fold_setting(
        '25%_labeled', labeled, test, npz_data, DEVICE,
        EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY, RANDOM_SEED, NUM_CLASSES, T_FRAMES
    )
    fold_results[fold]['100%_labeled'] = run_fold_setting(
        '100%_labeled', full_train, test, npz_data, DEVICE,
        EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY, RANDOM_SEED, NUM_CLASSES, T_FRAMES
    )

print(f'\nTotal wall time: {(time.time()-t_total)/60:.1f} min')

## 8 — Summary Table

In [ ]:
accs_25  = [fold_results[f]['25%_labeled']['accuracy']  for f in range(3)]
accs_100 = [fold_results[f]['100%_labeled']['accuracy'] for f in range(3)]
mean_25,  std_25  = np.mean(accs_25),  np.std(accs_25)
mean_100, std_100 = np.mean(accs_100), np.std(accs_100)

print('\n' + '='*65)
print(f'{"RESULTS — Widar_CNN_GRU (SenseFi)": ^65}')
print('='*65)
print(f'{"Fold":<6} {"Test Room":<12} {"CNN-GRU 25% labels":>20} {"CNN-GRU 100% labels":>21}')
print('-'*65)
for f in range(3):
    print(f'{f:<6} {FOLD_ROOMS[f]:<12} '
          f'{fold_results[f]["25%_labeled"]["accuracy"]:>19.2f}%  '
          f'{fold_results[f]["100%_labeled"]["accuracy"]:>19.2f}%')
print('-'*65)
print(f'{"Mean":<18} {mean_25:>19.2f}%  {mean_100:>19.2f}%')
print(f'{"Std":<18}  {std_25:>18.2f}%   {std_100:>18.2f}%')
print('='*65)
print()
print('All baselines (100% labels, LORO):')
print(f'  SVM (ours)          : 69.03% ± 7.76%')
print(f'  LeNet/CNN-5 (SenseFi): see results_lenet_baseline.json')
print(f'  CNN-GRU (this run)  : {mean_100:.2f}% ± {std_100:.2f}%')
print(f'  PI-SSL target (25%) : >89.50%')

## 9 — Save Results

In [ ]:
RESULTS_PATH = '/content/drive/MyDrive/pi-ssl/results_cnngru_baseline.json'

summary = {
    'method'       : 'Widar_CNN_GRU (SenseFi)',
    'source_paper' : 'Yang et al., Patterns 2023',
    'source_code'  : 'https://github.com/xyanchen/WiFi-CSI-Sensing-Benchmark',
    'adaptations'  : [
        'T_FRAMES changed from 22 to 20 to match our T_TARGET=20 preprocessing',
        'nn.Softmax removed from classifier; CrossEntropyLoss expects raw logits'
    ],
    'epochs'       : EPOCHS,
    'batch_size'   : BATCH_SIZE,
    'lr'           : LR,
    'weight_decay' : WEIGHT_DECAY,
    'labeled_ratio': LABELED_RATIO,
    'seed'         : RANDOM_SEED,
    'folds': {
        str(f): {
            'test_room'        : FOLD_ROOMS[f],
            'acc_25pct'        : fold_results[f]['25%_labeled']['accuracy'],
            'acc_100pct'       : fold_results[f]['100%_labeled']['accuracy'],
            'train_time_25pct' : fold_results[f]['25%_labeled']['train_time'],
            'train_time_100pct': fold_results[f]['100%_labeled']['train_time'],
        } for f in range(3)
    },
    'mean_acc_25pct'  : float(mean_25),
    'std_acc_25pct'   : float(std_25),
    'mean_acc_100pct' : float(mean_100),
    'std_acc_100pct'  : float(std_100),
}

with open(RESULTS_PATH, 'w') as fp:
    json.dump(summary, fp, indent=2)

print(f'Results saved to {RESULTS_PATH}')
print(json.dumps(summary, indent=2))

---
## Paper Note

In your paper write:

> "We adopt the Widar_CNN_GRU architecture from SenseFi [Yang et al., 2023],
> which processes each BVP frame independently through a shared CNN encoder
> and then models temporal dynamics with a single-layer GRU.
> We adapt T from 22 to 20 to match our preprocessing and remove the
> inference-time Softmax for compatibility with cross-entropy training.
> All other components are identical to the original implementation.
> We re-evaluate under our 3-fold LORO protocol."

**Next:** `baseline_simclr.ipynb` — generic SSL baseline (after PI-SSL encoder is designed).